# Chapter 7: Information Theory - Examples

Jupyter notebook demonstrating Shannon entropy, Kullback-Leibler divergence, mutual information, and source coding.

## 1. Shannon entropy

Compute entropy $H(X) = -\sum_i p_i \log_2 p_i$ for several distributions.

In [ ]:
import math

def entropy(probs):
    return -sum(p * math.log2(p) for p in probs if p > 0)

# Fair coin: uniform over 2 outcomes
print(f"H(fair coin)  = {entropy([0.5, 0.5]):.4f} bits  (max = 1)")

# Loaded coin: p=0.9
print(f"H(loaded coin)= {entropy([0.9, 0.1]):.4f} bits")

# Uniform 6-sided die
print(f"H(fair die)   = {entropy([1/6]*6):.4f} bits  (max = log2 6 = {math.log2(6):.4f})")

# Deterministic (no uncertainty)
print(f"H(constant)   = {entropy([1.0]):.4f} bits")

## 2. Conditional entropy and mutual information

$I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X)$.

In [ ]:
def joint_entropy(joint):
    """joint is a 2D list P(X=x, Y=y)."""
    H = 0
    for row in joint:
        for p in row:
            if p > 0:
                H -= p * math.log2(p)
    return H

def marginal(joint, axis):
    if axis == 0:  # marginal over Y -> P(X)
        return [sum(row) for row in joint]
    else:
        return [sum(joint[i][j] for i in range(len(joint))) for j in range(len(joint[0]))]

# Joint distribution: X = coin, Y = noisy copy, correctness q
q = 0.9
joint = [[q/2, (1-q)/2], [(1-q)/2, q/2]]
Px = marginal(joint, 0)
Py = marginal(joint, 1)
H_joint = joint_entropy(joint)
H_X = entropy(Px)
H_Y = entropy(Py)
MI = H_X + H_Y - H_joint
print(f"H(X) = {H_X:.4f}, H(Y) = {H_Y:.4f}, H(X,Y) = {H_joint:.4f}")
print(f"I(X;Y) = {MI:.4f} bits")

## 3. KL divergence

$D_{KL}(P||Q) = \sum_x P(x) \log (P(x)/Q(x))$, always nonnegative, zero iff $P=Q$.

In [ ]:
def kl(P, Q):
    return sum(p * math.log2(p/q) for p, q in zip(P, Q) if p > 0)

P = [0.5, 0.5]
Q1 = [0.5, 0.5]  # identical
Q2 = [0.6, 0.4]
Q3 = [0.9, 0.1]
print(f"D_KL(P||P)   = {kl(P, Q1):.4f}")
print(f"D_KL(P||Q2)  = {kl(P, Q2):.4f}")
print(f"D_KL(P||Q3)  = {kl(P, Q3):.4f}")
print(f"D_KL(Q3||P)  = {kl(Q3, P):.4f}  (not symmetric)")

## 4. Entropy of a message length / randomness testing

Compute the empirical entropy of a byte string; random data approaches 8 bits/byte.

In [ ]:
import math, random
from collections import Counter

def byte_entropy(data):
    counts = Counter(data)
    n = len(data)
    return -sum((c/n) * math.log2(c/n) for c in counts.values())

# Random 1KB (reproducible)
rng = random.Random(42)
rand_bytes = bytes(rng.randrange(256) for _ in range(1024))
print(f"Entropy of random bytes (1024) = {byte_entropy(rand_bytes):.4f} bits/byte")

# ASCII English text (lower-entropy)
text = b"the quick brown fox jumps over the lazy dog the quick brown fox"
print(f"Entropy of repeated English text = {byte_entropy(text):.4f} bits/byte")

## 5. Huffman coding / optimal prefix code

Optimal lossless source coding: average code length approaches the entropy.

In [ ]:
import heapq
from itertools import count

def huffman(probs):
    """Return dict symbol -> code using Huffman's algorithm.
    Heap items (priority, seq, children): leaves have a single symbol child."""
    seq = count()
    # leaves: (prob, order, [sym])
    heap = [(p, next(seq), [sym]) for sym, p in enumerate(probs)]
    heapq.heapify(heap)
    while len(heap) > 1:
        a = heapq.heappop(heap)
        b = heapq.heappop(heap)
        heapq.heappush(heap, (a[0] + b[0], next(seq), [a, b]))

    codes = {}

    def walk(node, prefix):
        children = node[2]
        if len(children) == 1 and not isinstance(children[0], tuple):
            codes[children[0]] = prefix
            return
        l, r = children
        walk(l, prefix + "0")
        walk(r, prefix + "1")

    walk(heap[0], "")
    return codes

probs = [0.5, 0.25, 0.125, 0.125]
codes = huffman(probs)
avg_len = sum(p * len(codes[i]) for i, p in enumerate(probs))
print("Huffman codes:", codes)
print(f"Average length = {avg_len:.4f} bits, entropy = {entropy(probs):.4f} bits")